In [35]:
import pandas as pd
h2_clean = pd.read_csv("https://raw.githubusercontent.com/lunettenoire/wellbeing_project/refs/heads/main/data/cleaned/df_h1.csv")
# On part de df_H1 qui contient les pays étudiés dans l'hypothèse 1 pour garder une cohérence

# Sélection des colonnes spécifiques
colonnes_a_garder = [
    'country_name', 
    'year', 
    'life_ladder', 
    'social_support', 
    'freedom_to_make_life_choices',
    'generosity', 
    'perceptions_of_corruption'
]

h2_clean = h2_clean[colonnes_a_garder]

# Remplacement des valeurs manquantes

**social_support**
· Vietnam : année 2017 manquante → remplacement par la moyenne des deux années précédentes et des deux années suivantes.\

**freedom_to_make_life_choices**

· Tadjikistan : 6 valeurs manquantes → exclusion pour cette variable. Sera fait dans un nouveau df

· Algérie : 1/7 → remplacement par la moyenne.

· Arabie Saoudite : 2/9 → remplacement par la moyenne.

· Chine : 2/8 → remplacement par la moyenne.

· Vietnam : 2/9 → remplacement par la moyenne.

**generosity**

· Algérie : 1/7 → remplacement par la médiane (présence de valeurs extrêmes).

· Islande : 1/8 → remplacement par la moyenne.

· Vietnam : 1/9 → remplacement par la moyenne.


In [36]:
# Etape 1 - social_support
annees_autour = [2015, 2016, 2018, 2019]
moyenne_vietnam = h2_clean[(h2_clean['country_name'] == 'Vietnam') & (h2_clean['year'].isin(annees_autour))]['social_support'].mean()
h2_clean.loc[(h2_clean['country_name'] == 'Vietnam') & (h2_clean['year'] == 2017),'social_support'] = moyenne_vietnam


In [37]:
#Etape 2: On remplace les valeurs manquantes de freedom[...] par les moyennes

pays_a_imputer = ['Algeria', 'Saudi Arabia', 'China', 'Vietnam']

for pays in pays_a_imputer:
    # 1. Calcul de la moyenne pour ce pays spécifique (en ignorant les NaN)
    moyenne_pays = h2_clean.loc[h2_clean['country_name'] == pays, 'freedom_to_make_life_choices'].mean()
    
    # 2. Remplissage des valeurs manquantes uniquement pour ce pays
    h2_clean.loc[(h2_clean['country_name'] == pays) & (h2_clean['freedom_to_make_life_choices'].isna()), 'freedom_to_make_life_choices'] = moyenne_pays

In [38]:
# Etape 3: Generosity

# Calcul de la médiane pour l'Algérie
mediane_algerie = h2_clean.loc[h2_clean['country_name'] == 'Algeria', 'generosity'].median()

h2_clean.loc[(h2_clean['country_name'] == 'Algeria') & (h2_clean['generosity'].isna()),'generosity'] = mediane_algerie

# Liste des pays concernés par la moyenne
pays_moyenne = ['Iceland', 'Vietnam']

for pays in pays_moyenne:
    # Calcul de la moyenne spécifique au pays
    moyenne_pays = h2_clean.loc[h2_clean['country_name'] == pays, 'generosity'].mean()
    
    # Remplacement du NaN
    h2_clean.loc[(h2_clean['country_name'] == pays) & (h2_clean['generosity'].isna()),'generosity'] = moyenne_pays

In [39]:
h2_clean.to_csv("h2_clean_wo_corruption.csv", index=False)

In [40]:
# Reste à faire: Supprimer le Tajikistan dans un DF spécial pour freedom_to_make_life_choices
# Suppression des lignes où country_name est "Tajikistan"
h2_clean_freedom = h2_clean[h2_clean['country_name'] != 'Tajikistan']

h2_clean_freedom.to_csv("h2_clean_freedom.csv", index=False)

**perceptions_of_corruption**
Algérie, Cambodia, Niger, Singapore, Uzbekistan   = 1 NaN
Chine, Libya, Vietnam  = 2 NaN
**Yemen = ⅖ NaN
Egypt, Jordan = 6/9 NaN
Bahrain, Kuwait = 6 NaN -> aucune valeur
Maldives, Qatar = 1/1 NaN
Saudi Arabia, United Arab Emirates  = 9/9 NaN
Turkmenistan = 5/5 NaN**

In [41]:
# création du df où on supprime les NaN de Perceptions of corruptions

# 1. Création du nouveau DataFrame
h2_clean_corruption = h2_clean.copy()

# --- SUPPRESSIONS MASSIVES (Par Pays) ---
pays_a_supprimer = ['China', 'Egypt', 'Jordan', 'Saudi Arabia', 'United Arab Emirates']
h2_clean_corruption = h2_clean_corruption[~h2_clean_corruption['country_name'].isin(pays_a_supprimer)]

# --- SUPPRESSIONS CIBLÉES (Par Pays et Année) ---
# On définit une liste de tuples (pays, année) pour automatiser la suppression
lignes_a_retirer = [
    ('Algeria', 2016),
    ('Niger', 2023),
    ('Singapore', 2022),
    ('Vietnam', 2015)
]

for pays, annee in lignes_a_retirer:
    h2_clean_corruption = h2_clean_corruption.drop(
        h2_clean_corruption[(h2_clean_corruption['country_name'] == pays) & (h2_clean_corruption['year'] == annee)].index
    )

# --- MODIFICATIONS (MOYENNES) ---

# Cambodge 2018 (moyenne 2017 et 2019)
moy_cambodge = h2_clean_corruption.loc[
    (h2_clean_corruption['country_name'] == 'Cambodia') & (h2_clean_corruption['year'].isin([2017, 2019])), 
    'perceptions_of_corruption'
].mean()
h2_clean_corruption.loc[(h2_clean_corruption['country_name'] == 'Cambodia') & (h2_clean_corruption['year'] == 2018), 'perceptions_of_corruption'] = moy_cambodge

# Ouzbékistan 2016 (moyenne 2015 et 2017)
moy_uzb = h2_clean_corruption.loc[
    (h2_clean_corruption['country_name'] == 'Uzbekistan') & (h2_clean_corruption['year'].isin([2015, 2017])), 
    'perceptions_of_corruption'
].mean()
h2_clean_corruption.loc[(h2_clean_corruption['country_name'] == 'Uzbekistan') & (h2_clean_corruption['year'] == 2016), 'perceptions_of_corruption'] = moy_uzb

# Vietnam 2017 (moyenne 2016 et 2018)
moy_viet = h2_clean_corruption.loc[
    (h2_clean_corruption['country_name'] == 'Vietnam') & (h2_clean_corruption['year'].isin([2016, 2018])), 
    'perceptions_of_corruption'
].mean()
h2_clean_corruption.loc[(h2_clean_corruption['country_name'] == 'Vietnam') & (h2_clean_corruption['year'] == 2017), 'perceptions_of_corruption'] = moy_viet

# 2. Réinitialisation de l'index pour avoir un DF propre
h2_clean_corruption = h2_clean_corruption.reset_index(drop=True)

h2_clean_corruption.to_csv("h2_clean_corruption.csv", index=False)